In [29]:
!pip install biopython

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Extracting Data from Dataset
Getting the data from the dataset and then converting it to a dataframe so it is easy to access

In [31]:
db = load_dataset("tattabio/modac_paralogy_bigene")
print(db)

DatasetDict({
    train: Dataset({
        features: ['ID1', 'Seq1', 'ID2', 'Seq2'],
        num_rows: 1492
    })
})


In [32]:
pos_data = db['train']

pos_data = pos_data.to_pandas()
pos_data['label'] = 1
print(pos_data)

         ID1                                               Seq1     ID2  \
0     E0I7Z3  MSDTNSIIRFERVTKRYDNGSPVLSDVSFEIERGKFYTLLGPSGCG...  E0I7Z2   
1     D9PZK5  MILNAGMLSSGGVDMEYIRLEDVWKTYRTKNVTATPLRGLNMNVDK...  D9PZK6   
2     C4LAU3  MLLAEKLQTRRQGRLFEFSLQLQPGEIGLLLGRSGSGKSTLLEMLA...  C4LAU4   
3     C4LAH6  MNAIEIHNLQCGYQEQAILQNVSFVLEERKILALLGPSGCGKTTLL...  C4LAH7   
4     C4L8Q6  MHDIEIRLCWSRSEFQLDVALQLPGQGVSALFGPSGCGKTTCLRAI...  C4L8Q7   
...      ...                                                ...     ...   
1487  Q1LU07  MSITIENVSKFFDNTQVLNNISLDINSGQMVALLGPSGSGKTTLLR...  Q1LU08   
1488  C7MHE4  MNSVDLENVTKIYSGSTPSVDDVSLTVGDGEFFTLLGPSGCGKSTT...  C7MHE5   
1489  C7MHQ3  MITFDDITVRFGQFTALPSLSLNIDEGEFFTLLGPSGCGKSTALRT...  C7MHQ2   
1490  Q2CC48  MTELSLRGLTKRFGNHTAVDDVTLDVPDGSFICLLGPSGCGKTTLL...  Q2CC47   
1491  Q2CJ49  MASVEISGLRKLYADVVALEDINLSIPTGSFYTLLGPSGCGKTTLL...  Q2CJ48   

                                                   Seq2  label  
0     MPSRTRLRMSDKSRNWYLIPYTAWIVLF

# Creating the extra N rows
This is so that there is negative data for the model to see as well so that it can make more accurate predictions later on

In [33]:
neg_data = pos_data.copy()
shuffle = pos_data[['ID2', 'Seq2']].sample(frac=1).reset_index(drop=True)
neg_data['ID2'] = shuffle['ID2']
neg_data['Seq2'] = shuffle['Seq2']
neg_data['label'] = 0
print(neg_data)

         ID1                                               Seq1     ID2  \
0     E0I7Z3  MSDTNSIIRFERVTKRYDNGSPVLSDVSFEIERGKFYTLLGPSGCG...  B0U9S3   
1     D9PZK5  MILNAGMLSSGGVDMEYIRLEDVWKTYRTKNVTATPLRGLNMNVDK...  E7RD21   
2     C4LAU3  MLLAEKLQTRRQGRLFEFSLQLQPGEIGLLLGRSGSGKSTLLEMLA...  F4CZV8   
3     C4LAH6  MNAIEIHNLQCGYQEQAILQNVSFVLEERKILALLGPSGCGKTTLL...  F3KQI3   
4     C4L8Q6  MHDIEIRLCWSRSEFQLDVALQLPGQGVSALFGPSGCGKTTCLRAI...  F0J5K3   
...      ...                                                ...     ...   
1487  Q1LU07  MSITIENVSKFFDNTQVLNNISLDINSGQMVALLGPSGSGKTTLLR...  E8TB53   
1488  C7MHE4  MNSVDLENVTKIYSGSTPSVDDVSLTVGDGEFFTLLGPSGCGKSTT...  D7A6H7   
1489  C7MHQ3  MITFDDITVRFGQFTALPSLSLNIDEGEFFTLLGPSGCGKSTALRT...  B7TYM6   
1490  Q2CC48  MTELSLRGLTKRFGNHTAVDDVTLDVPDGSFICLLGPSGCGKTTLL...  D3AF26   
1491  Q2CJ49  MASVEISGLRKLYADVVALEDINLSIPTGSFYTLLGPSGCGKTTLL...  A5G1M7   

                                                   Seq2  label  
0     MSVASLSPASAPALAARPGLSVRLGRAA

# Joining the positive and negative data to get a final database

In [34]:
final_db = pd.concat([pos_data, neg_data], axis=0)
print(final_db)

         ID1                                               Seq1     ID2  \
0     E0I7Z3  MSDTNSIIRFERVTKRYDNGSPVLSDVSFEIERGKFYTLLGPSGCG...  E0I7Z2   
1     D9PZK5  MILNAGMLSSGGVDMEYIRLEDVWKTYRTKNVTATPLRGLNMNVDK...  D9PZK6   
2     C4LAU3  MLLAEKLQTRRQGRLFEFSLQLQPGEIGLLLGRSGSGKSTLLEMLA...  C4LAU4   
3     C4LAH6  MNAIEIHNLQCGYQEQAILQNVSFVLEERKILALLGPSGCGKTTLL...  C4LAH7   
4     C4L8Q6  MHDIEIRLCWSRSEFQLDVALQLPGQGVSALFGPSGCGKTTCLRAI...  C4L8Q7   
...      ...                                                ...     ...   
1487  Q1LU07  MSITIENVSKFFDNTQVLNNISLDINSGQMVALLGPSGSGKTTLLR...  E8TB53   
1488  C7MHE4  MNSVDLENVTKIYSGSTPSVDDVSLTVGDGEFFTLLGPSGCGKSTT...  D7A6H7   
1489  C7MHQ3  MITFDDITVRFGQFTALPSLSLNIDEGEFFTLLGPSGCGKSTALRT...  B7TYM6   
1490  Q2CC48  MTELSLRGLTKRFGNHTAVDDVTLDVPDGSFICLLGPSGCGKTTLL...  D3AF26   
1491  Q2CJ49  MASVEISGLRKLYADVVALEDINLSIPTGSFYTLLGPSGCGKTTLL...  A5G1M7   

                                                   Seq2  label  
0     MPSRTRLRMSDKSRNWYLIPYTAWIVLF

# Creating the features for each sequence

In [35]:
def get_features(seq):
  # Amino acids composition
  analyzed_seq = ProteinAnalysis(seq.replace("X","").replace("Z",""))
  aa_comp = list(analyzed_seq.amino_acids_percent.values())

  # Dipeptide frequency
  aa = 'ACDEFGHIKLMNPQRSTVWY'
  dipeptides = {a+b:0 for a in aa for b in aa}

  for i in range(len(seq) - 1):
    pair = seq[i:i+2]
    if pair in dipeptides:
      dipeptides[pair] += 1

  dipeptide_freq = [count / (len(seq) - 1) for count in dipeptides.values()]

  # Physicochemical descriptors
  phys_desc = [
      analyzed_seq.gravy(),
      analyzed_seq.isoelectric_point(),
      analyzed_seq.molecular_weight(),
      analyzed_seq.aromaticity()
  ]

  return aa_comp + dipeptide_freq + phys_desc

In [36]:
f1_list = []
f2_list = []

for index, row in final_db.iterrows():
    f1 = get_features(row['Seq1'])
    f2 = get_features(row['Seq2'])

    f1_list.append(f1)
    f2_list.append(f2)


aa_names = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
dipep_names = [f"dp_{i}" for i in range(400)]
phys_names = ['gravy', 'isoelectric_point', 'mol_weight', 'aromaticity']

header = aa_names + dipep_names + phys_names

df_features1 = pd.DataFrame(f1_list, columns=[f"s1_{name}" for name in header])
df_features2 = pd.DataFrame(f2_list, columns=[f"s2_{name}" for name in header])

final_db = pd.concat([final_db.reset_index(drop=True), df_features1, df_features2], axis=1)

print(final_db)

         ID1                                               Seq1     ID2  \
0     E0I7Z3  MSDTNSIIRFERVTKRYDNGSPVLSDVSFEIERGKFYTLLGPSGCG...  E0I7Z2   
1     D9PZK5  MILNAGMLSSGGVDMEYIRLEDVWKTYRTKNVTATPLRGLNMNVDK...  D9PZK6   
2     C4LAU3  MLLAEKLQTRRQGRLFEFSLQLQPGEIGLLLGRSGSGKSTLLEMLA...  C4LAU4   
3     C4LAH6  MNAIEIHNLQCGYQEQAILQNVSFVLEERKILALLGPSGCGKTTLL...  C4LAH7   
4     C4L8Q6  MHDIEIRLCWSRSEFQLDVALQLPGQGVSALFGPSGCGKTTCLRAI...  C4L8Q7   
...      ...                                                ...     ...   
2979  Q1LU07  MSITIENVSKFFDNTQVLNNISLDINSGQMVALLGPSGSGKTTLLR...  E8TB53   
2980  C7MHE4  MNSVDLENVTKIYSGSTPSVDDVSLTVGDGEFFTLLGPSGCGKSTT...  D7A6H7   
2981  C7MHQ3  MITFDDITVRFGQFTALPSLSLNIDEGEFFTLLGPSGCGKSTALRT...  B7TYM6   
2982  Q2CC48  MTELSLRGLTKRFGNHTAVDDVTLDVPDGSFICLLGPSGCGKTTLL...  D3AF26   
2983  Q2CJ49  MASVEISGLRKLYADVVALEDINLSIPTGSFYTLLGPSGCGKTTLL...  A5G1M7   

                                                   Seq2  label       s1_A  \
0     MPSRTRLRMSDKSRNW

# Creating the Model and Hyperparameter Tuning

In [38]:
X = final_db.iloc[:, 5:]
y = final_db['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, solver='lbfgs'))
])

param_grid = {
    'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'logreg__penalty': ['l2']
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('logreg',
                                        LogisticRegression(max_iter=1000))]),
             param_grid={'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'logreg__penalty': ['l2']},
             scoring='accuracy')

# Checking Model Performance

In [41]:
y_pred = grid_search.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

print()

recall = recall_score(y_test, y_pred)
print("Recall: ", recall)

print()

f1 = f1_score(y_test, y_pred)
print("F1 score: ", f1)


Accuracy:  0.19430485762144054

Recall:  0.15625

F1 score:  0.1721170395869191
